# 4. Predictive / Forecasting Tools

*Estimated time to run notebook: about 8-10 min*

**Goal:** Use a DataRobot forecast model deployment as a tool.

**Key Concept:**
LLMs cannot predict the future, but DataRobot can. In this notebook, we write a **Tool Client** function. This wrapper:
1.  **Defines a Schema:** Forces the LLM to extract structured inputs (e.g., a specific date and item).
2.  **Activates:** Triggers only when the user asks a forward-looking question.
3.  **Predicts:** Queries a deployed Time Series model to return an accurate forecast number to the chat.

**Note:** Set `MCP_DEPLOYMENT_ID` in `.env` using an existing deployed MCP server, or create one in **Notebook 0 - MCP Server Setup (Optional)**.

In [4]:
import os
from dotenv import load_dotenv
from pprint import pprint
import datarobot as dr
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.mcp import MCPServerStreamableHTTP

# 1. Initialize client
load_dotenv(override=True)
dr_client = dr.Client()

MCP_DEPLOYMENT_ID = os.getenv("MCP_DEPLOYMENT_ID")
if not MCP_DEPLOYMENT_ID:
    raise ValueError(
        "MCP_DEPLOYMENT_ID is not set. Add an existing MCP deployment ID to .env, "
        "or create one in Notebook 0."
    )
PROMPT_TEMPLATE_ID = os.getenv("PROMPT_TEMPLATE_ID")
FORECAST_DEPLOYMENT_ID = os.getenv("FORECAST_DEPLOYMENT_ID")
SCORING_DATASET_ID = os.getenv("SCORING_DATASET_ID")

# 2. Configure the tool connection (MCP)
server = MCPServerStreamableHTTP(
    f"{dr_client.endpoint}/deployments/{MCP_DEPLOYMENT_ID}/directAccess/mcp",
    headers={
        "Authorization": f"Bearer {dr_client.token}",
        "x-datarobot-api-token": dr_client.token,
    },
    max_retries=3,
    timeout=60.0,
)

# 3. Configure the model
# Using the same Azure GPT-5 configuration via the DataRobot Gateway
MODEL_NAME = os.getenv("MODEL_NAME")
llm = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        api_key=dr_client.token, 
        base_url=dr_client.endpoint + "/genai/llmgw"
    ),
)

# 4. Define the agent
# We update the prompt to ensure the agent understands its role is forecasting

system_prompt = """
You are an ERCOT market analyst specializing in ERCOT day-ahead market hub prices.
Forecasting workflow (use only tools that exist on the server):
- To score an **AI Catalog dataset** against a **deployment**, use **predict_by_ai_catalog** or
  **predict_by_ai_catalog_rt**.
- Use **get_deployment_features** or **get_deployment_info** if you need the datetime / series columns
  or to confirm time-series settings before scoring.

After scoring, summarize predictions for the user’s requested date range (e.g. all hours in that day).
"""

agent = Agent(model=llm, toolsets=[server], system_prompt=system_prompt)

# 5. Run the forecast
# The agent will detect the forecast tool exposed by the MCP server and use it
async with server:
    response = await agent.run(f"Please generate a forecast using the scoring dataset ID {SCORING_DATASET_ID} and the deployment iD {FORECAST_DEPLOYMENT_ID} and find the forecast for 2025-10-27 whole day.")
    pprint(response.output)

In [5]:
# Smoke test: ask the agent (via MCP tools) what forecast models/deployments exist for your default use case.
USE_CASE_ID = os.getenv("DATAROBOT_DEFAULT_USE_CASE")
async with server:
    response = await agent.run(f"Please list the forecasting models or deployments you currently have access to (use case {USE_CASE_ID}).")
    pprint(response.output)

## Using DR managed prompts 

In [6]:
from dotenv import load_dotenv
from pprint import pprint
import datarobot as dr
from datarobot.models.genai.prompt_template import PromptTemplate
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.mcp import MCPServerStreamableHTTP

# 1. Initialize client
load_dotenv()
dr_client = dr.Client()

# 2. Configure the tool connection (MCP)
MCP_DEPLOYMENT_ID = os.getenv("MCP_DEPLOYMENT_ID")
if not MCP_DEPLOYMENT_ID:
    raise ValueError(
        "MCP_DEPLOYMENT_ID is not set. Add an existing MCP deployment ID to .env, "
        "or create one in Notebook 0."
    )
server = MCPServerStreamableHTTP(
    f"{dr_client.endpoint}/deployments/{MCP_DEPLOYMENT_ID}/directAccess/mcp",
    headers={
        "Authorization": f"Bearer {dr_client.token}",
        "x-datarobot-api-token": dr_client.token,
    },
    max_retries=3,
    timeout=60.0,
)

# --- NEW STEP: PREPARE PROMPT CONTEXT ---
# We fetch the deployment details so we can inject the real name into the prompt
deployment = dr.Deployment.get(MCP_DEPLOYMENT_ID)
deployment_name = deployment.label  # e.g., "Sales Forecast Production"

# 3. Fetch and render system prompt
PROMPT_TEMPLATE_ID = os.getenv("PROMPT_TEMPLATE_ID")
FORECAST_DEPLOYMENT_ID = os.getenv("FORECAST_DEPLOYMENT_ID")
SCORING_DATASET_ID = os.getenv("SCORING_DATASET_ID")

# OPTIONAL: Set to specific version like "v1", or None for latest
PROMPT_VERSION_ID = "v1" 

print(f"--- Setting up Agent for Deployment: {deployment_name} ---")

# Fetch template
template = PromptTemplate.get(PROMPT_TEMPLATE_ID)

# Determine version (smart logic)
target_version = None
if PROMPT_VERSION_ID:
    versions = template.list_versions()
    search_str = str(PROMPT_VERSION_ID).lower().replace("v", "")
    for v in versions:
        if v.id == PROMPT_VERSION_ID:
            target_version = v
            break
        if hasattr(v, 'version') and str(v.version) == search_str:
            target_version = v
            break
    if not target_version:
        raise ValueError(f"Version '{PROMPT_VERSION_ID}' not found.")
else:
    target_version = template.get_latest_version()

print(f"Using Prompt Version: v{getattr(target_version, 'version', '?')} (ID: {target_version.id})")

# Render prompt with dynamic variables
try:
    system_prompt = target_version.render(
        variables={
            "company_name": "DataRobot Forecasting Inc.",
            "forecast_deployment": FORECAST_DEPLOYMENT_ID,
            "scoring_dataset": SCORING_DATASET_ID,
        }
    )
except Exception as e:
    print(
        "Error rendering prompt. Ensure the prompt template variables match "
        "company_name, forecast_deployment, and scoring_dataset."
    )
    raise e

# 4. Configure the model
MODEL_NAME = os.getenv("MODEL_NAME")
llm = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        api_key=dr_client.token, 
        base_url=dr_client.endpoint + "/genai/llmgw"
    ),
)

# 5. Define the agent
# Now using the dynamic 'system_prompt' from DataRobot
agent = Agent(model=llm, toolsets=[server], system_prompt=system_prompt)

# 6. Run the forecast
async with server:
    print("\n--- Running Forecast Request ---")
    response = await agent.run("Generate forecasts for 2025-10-27 for all hubs and display as tables")
    pprint(response.output)